In [ ]:
# ensemble_benchmark.ipynb
# Fast per-class ensemble strategy evaluation.
# First run: scores are computed and cached to disk per class.
# Subsequent runs: load cache (<1s per class), run experiments immediately.

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, pandas as pd
from pathlib import Path
from PIL import Image
from scipy.ndimage import median_filter, label as cc_label
from sklearn.metrics import average_precision_score
from collections import defaultdict
from tqdm.auto import tqdm
import torchvision.transforms as T
import torchvision.models as _tvm
import timm, inspect, random, warnings, re as _re, dataclasses as _dc, json
from dataclasses import dataclass as _dataclass
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Paths — keep in sync with main notebook ───────────────────────────────────
DATA_ROOT = Path('/content/birefnet')
UNET_DIR  = Path('/content/drive/MyDrive/ADL/challenge/outputs_v2/models')
CACHE_DIR = Path('/content/drive/MyDrive/ADL/challenge/bm_cache')

# ══════════ CONFIGURE HERE ═══════════════════════════════════════════════════
# Subset of class_* folders to benchmark, or None to run all.
TARGET_CLASSES = ['class_03']   # e.g. ['class_01', 'class_05']

# Must match main notebook exactly
VAL_FRAC      = 0.20
VAL_SEED      = 42
BACKBONE      = 'vit_base_patch14_reg4_dinov2.lvd142m'
IMAGE_SIZE    = 518
PATCH_SIZE    = 14
GRID_SIZE     = IMAGE_SIZE // PATCH_SIZE   # 37
LAYERS        = [3, 6, 9, 11]
FEAT_DIM_RAW  = 768 * len(LAYERS)          # 3072
PCA_DIM       = 384
KNN_K         = 3
SMOOTH_SIGMA  = 0.5
SCORE_CHUNK   = 2048
BATCH_SIZE    = 64
CALIB_HIGH    = 99
CALIB_N       = 256
CORESET_SIZE  = 50_000
UNET_IMG_SIZE = 512
ENS_USE_TTA   = True

# Experiment params
ALPHA_STEPS  = 41
SWEEP_FRAC   = 0.20
GAMMA_VALUES = [0.5, 0.7, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0]
MORPH_SIZES  = [3, 5, 7]
MIN_AREAS    = [0, 25, 50, 100, 200]
# ═════════════════════════════════════════════════════════════════════════════

print(f'Device: {DEVICE}')

Device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CACHE_DIR.mkdir(parents=True, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import shutil

GDRIVE_ID = '1lCqjjtKOhqoU3M5-4ktgiVlWnp6IFDYC'

if not DATA_ROOT.exists():
    zip_path = Path('/content/birefnet.zip')
    print('Downloading ...')
    !gdown {GDRIVE_ID} -O {zip_path}
    print('Unzipping ...')
    tmp = Path('/content/_unzip_tmp')
    !unzip -q {zip_path} -d {tmp}
    children = list(tmp.iterdir())
    if len(children) == 1 and children[0].is_dir():
        shutil.move(str(children[0]), str(DATA_ROOT))
        tmp.rmdir()
    else:
        shutil.move(str(tmp), str(DATA_ROOT))
    zip_path.unlink(missing_ok=True)
    print(f'Ready at {DATA_ROOT}')
else:
    print(f'Already present: {DATA_ROOT}')

# Discover classes exactly as the main notebook does
all_classes = sorted(d.name for d in DATA_ROOT.iterdir()
                     if d.is_dir() and d.name.startswith('class_'))
classes = TARGET_CLASSES if TARGET_CLASSES else all_classes
print(f'Classes ({len(all_classes)} total, benchmarking {len(classes)}): {classes}')

Already present: /content/birefnet
Classes (8 total, benchmarking 1): ['class_03']


In [ ]:
# ── Data helpers ──────────────────────────────────────────────────────────────
_tfm = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class _DS(torch.utils.data.Dataset):
    def __init__(self, paths): self.paths = list(paths)
    def __len__(self):         return len(self.paths)
    def __getitem__(self, i):  return _tfm(Image.open(self.paths[i]).convert('RGB'))

def load_mask(mask_path):
    m = np.array(Image.open(mask_path).convert('L').resize((224, 224), Image.NEAREST))
    return (m > 0).astype(np.float32)

def val_items_for(cls):
    """Returns [(img_path, mask_path_or_None, atype)] — matches main notebook val split."""
    good_dir = DATA_ROOT / cls / 'train' / 'good'
    all_good = sorted(good_dir.glob('*.png'))
    assert all_good, f'No images found in {good_dir}'
    rng      = np.random.default_rng(VAL_SEED)
    idx      = rng.permutation(len(all_good))
    n_val    = max(1, int(len(all_good) * VAL_FRAC))
    val_good = [all_good[i] for i in sorted(idx[:n_val])]

    items    = [(p, None, 'good') for p in val_good]
    trn_root = DATA_ROOT / cls / 'train'
    gt_root  = DATA_ROOT / cls / 'ground_truth_train'
    for anom_dir in sorted(trn_root.iterdir()):
        if not anom_dir.name.startswith('anomaly_'):
            continue
        gd = gt_root / anom_dir.name
        for p in sorted(anom_dir.glob('*.png')):
            mp = gd / p.name
            if mp.exists():
                items.append((p, mp, anom_dir.name))
    return items

def train_normal_paths_for(cls):
    """80% split of good images — matches main notebook bank-building split."""
    good_dir = DATA_ROOT / cls / 'train' / 'good'
    all_good = sorted(good_dir.glob('*.png'))
    assert all_good, f'No images found in {good_dir}. Check TARGET_CLASSES uses class_* names.'
    rng      = np.random.default_rng(VAL_SEED)
    idx      = rng.permutation(len(all_good))
    n_val    = max(1, int(len(all_good) * VAL_FRAC))
    return [str(all_good[i]) for i in sorted(idx[n_val:])]

# ── PatchCore helpers ─────────────────────────────────────────────────────────
class GPUPCA:
    def __init__(self):
        self.mean_ = self.components_ = None

    @torch.no_grad()
    def fit(self, X_np):
        X                = torch.from_numpy(X_np.astype(np.float32)).to(DEVICE)
        self.mean_       = X.mean(0)
        _, _, V          = torch.pca_lowrank(X - self.mean_, q=PCA_DIM, niter=4)
        self.components_ = V
        return self

    @torch.no_grad()
    def transform(self, X_np):
        X = torch.from_numpy(X_np.astype(np.float32)).to(DEVICE)
        return ((X - self.mean_) @ self.components_).cpu().numpy()

def build_dino():
    try:
        m = timm.create_model(BACKBONE, pretrained=True, num_classes=0)
    except Exception:
        m = timm.create_model(BACKBONE.replace('_reg4', ''), pretrained=True, num_classes=0)
    for p in m.parameters():
        p.requires_grad_(False)
    sig         = inspect.signature(m.get_intermediate_layers)
    m._new_api  = 'return_class_token' in sig.parameters
    m._n_blocks = len(m.blocks)
    return m.to(DEVICE).eval()

@torch.no_grad()
def extract_batch(batch_t, dino):
    """batch_t: (B,3,518,518) -> (B, GRID^2, FEAT_DIM_RAW) float32"""
    if dino._new_api:
        outs   = dino.get_intermediate_layers(batch_t, n=LAYERS, reshape=False,
                                               return_class_token=False, norm=True)
        tokens = [o[:, -GRID_SIZE * GRID_SIZE:, :] for o in outs]
    else:
        outs   = dino.get_intermediate_layers(batch_t, n=dino._n_blocks)
        tokens = [outs[i][:, -GRID_SIZE * GRID_SIZE:, :] for i in LAYERS]
    return torch.cat(tokens, dim=-1).float()

def build_banks_for(cls_list, dino):
    """Build feature banks and fit PCA for the given classes. Returns (pca, banks dict)."""
    # Phase 1: per-batch coreset extraction
    raw_coresets = {}
    for cls in cls_list:
        train_paths = train_normal_paths_for(cls)
        n_batches   = max(1, (len(train_paths) + BATCH_SIZE - 1) // BATCH_SIZE)
        keep_pb     = max(1, CORESET_SIZE // n_batches)
        loader      = torch.utils.data.DataLoader(_DS(train_paths), batch_size=BATCH_SIZE, num_workers=0)
        chunks      = []
        with torch.no_grad():
            for batch in tqdm(loader, desc=f'{cls} feats', leave=False):
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(DEVICE.type == 'cuda')):
                    feats = extract_batch(batch.to(DEVICE), dino).float()
                p = feats.cpu().numpy().reshape(-1, FEAT_DIM_RAW)
                if len(p) > keep_pb:
                    p = p[np.random.choice(len(p), keep_pb, replace=False)]
                chunks.append(p)
        patches = np.concatenate(chunks);  del chunks
        if len(patches) > CORESET_SIZE:
            patches = patches[np.random.choice(len(patches), CORESET_SIZE, replace=False)]
        raw_coresets[cls] = patches
        print(f'  {cls}: {len(train_paths)} imgs -> {len(patches):,} coreset patches')

    # Phase 2: fit global PCA on a cross-class subsample
    combined = np.concatenate(list(raw_coresets.values()))
    n_sample = min(200_000, len(combined))
    idx      = np.random.choice(len(combined), n_sample, replace=False)
    print(f'Fitting PCA {FEAT_DIM_RAW}->{PCA_DIM} on {n_sample:,} patches ...', end=' ')
    pca = GPUPCA().fit(combined[idx]);  del combined
    print('done')

    # Phase 3: project coresets to PCA space -> GPU banks
    banks = {}
    for cls in cls_list:
        p          = pca.transform(raw_coresets.pop(cls)).astype(np.float32)
        banks[cls] = torch.from_numpy(p).to(DEVICE)
        print(f'  {cls}: bank {banks[cls].shape}')
    return pca, banks

def _gauss_kernel(sigma):
    ks = int(6 * sigma + 1) | 1
    x  = torch.arange(ks, dtype=torch.float32) - ks // 2
    g  = torch.exp(-x ** 2 / (2 * sigma ** 2));  g /= g.sum()
    return g.outer(g).view(1, 1, ks, ks).to(DEVICE)

_GAUSS     = _gauss_kernel(SMOOTH_SIGMA)
_GAUSS_PAD = _GAUSS.shape[-1] // 2

@torch.no_grad()
def score_patches(feat_ND, bank):
    q       = torch.from_numpy(feat_ND).to(DEVICE)
    bank_sq = (bank * bank).sum(1)
    out     = []
    for i in range(0, len(q), SCORE_CHUNK):
        qi = q[i:i + SCORE_CHUNK]
        d  = ((qi * qi).sum(1, keepdim=True) + bank_sq - 2.0 * (qi @ bank.T)).clamp(min=0)
        out.append(torch.topk(d, KNN_K, dim=1, largest=False).values.mean(1))
    return torch.cat(out).cpu().numpy()

def make_heatmap(scores_N):
    """(GRID^2,) -> (224,224) float32"""
    t = torch.from_numpy(scores_N.reshape(1, 1, GRID_SIZE, GRID_SIZE).astype(np.float32)).to(DEVICE)
    t = F.conv2d(t, _GAUSS, padding=_GAUSS_PAD)
    return F.interpolate(t, (224, 224), mode='bilinear', align_corners=False).squeeze().cpu().numpy()

@torch.no_grad()
def score_paths_pc(paths, dino, pca, bank, calib_hi):
    """Returns list of (224,224) calibrated float32 heatmaps."""
    heatmaps = []
    loader   = torch.utils.data.DataLoader(_DS(paths), batch_size=BATCH_SIZE, num_workers=0)
    G        = GRID_SIZE * GRID_SIZE
    for bi, batch in enumerate(tqdm(loader, leave=False, desc='PC score')):
        b_n = batch.shape[0]
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(DEVICE.type == 'cuda')):
            feats = extract_batch(batch.to(DEVICE), dino).float()
        fp = pca.transform(feats.cpu().numpy().reshape(-1, FEAT_DIM_RAW))
        sc = score_patches(fp, bank)
        for j in range(b_n):
            hm = make_heatmap(sc[j * G:(j + 1) * G])
            heatmaps.append(np.clip(hm / (calib_hi + 1e-8), 0., 1.).astype(np.float32))
    return heatmaps

def calib_hi_for(train_paths, dino, pca, bank):
    sample  = random.sample(train_paths, min(CALIB_N, len(train_paths)))
    loader  = torch.utils.data.DataLoader(_DS(sample), batch_size=BATCH_SIZE, num_workers=0)
    maxvals = []
    G       = GRID_SIZE * GRID_SIZE
    for batch in loader:
        b_n = batch.shape[0]
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(DEVICE.type == 'cuda')):
            feats = extract_batch(batch.to(DEVICE), dino).float()
        fp = pca.transform(feats.cpu().numpy().reshape(-1, FEAT_DIM_RAW))
        sc = score_patches(fp, bank)
        for j in range(b_n):
            maxvals.append(make_heatmap(sc[j * G:(j + 1) * G]).max())
    return float(np.percentile(maxvals, CALIB_HIGH))

# ── UNet helpers (copied from main notebook) ──────────────────────────────────
@_dataclass(frozen=True)
class _MCfg:
    encoder: str = 'resnet34'; pretrained: bool = True
    view_count: int = 5;       view_embed_dim: int = 64
    base_channels: int = 256;  use_attention_gates: bool = True

class _CBA(nn.Module):
    def __init__(self, i, o, k=3):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(i, o, k, padding=k//2, bias=False),
                                 nn.BatchNorm2d(o), nn.SiLU(inplace=True))
    def forward(self, x): return self.net(x)

class _FiLM(nn.Module):
    def __init__(self, cd, fc):
        super().__init__()
        self.to_gamma = nn.Linear(cd, fc); self.to_beta = nn.Linear(cd, fc)
    def forward(self, x, c):
        g = self.to_gamma(c).unsqueeze(-1).unsqueeze(-1)
        b = self.to_beta(c).unsqueeze(-1).unsqueeze(-1)
        return x * (1 + g) + b

class _AG(nn.Module):
    def __init__(self, gc, xc, ic):
        super().__init__()
        self.W_g = nn.Conv2d(gc, ic, 1); self.W_x = nn.Conv2d(xc, ic, 1, bias=False)
        self.psi = nn.Conv2d(ic, 1, 1)
    def forward(self, g, x):
        gu = F.interpolate(self.W_g(g), x.shape[-2:], mode='bilinear', align_corners=False)
        return x * torch.sigmoid(self.psi(F.relu(gu + self.W_x(x), inplace=True)))

class _Up(nn.Module):
    def __init__(self, ic, sc, oc, cd, attn=True):
        super().__init__()
        self.attn  = _AG(ic, sc, max(1, sc//2)) if attn else None
        self.conv1 = _CBA(ic+sc, oc); self.conv2 = _CBA(oc, oc)
        self.film  = _FiLM(cd, oc)
    def forward(self, x, sk, c):
        if self.attn: sk = self.attn(x, sk)
        x = F.interpolate(x, sk.shape[-2:], mode='bilinear', align_corners=False)
        return self.conv2(self.film(self.conv1(torch.cat([x, sk], 1)), c))

class _Enc(nn.Module):
    def __init__(self, name='resnet34'):
        super().__init__()
        if name == 'resnet18':
            m = _tvm.resnet18(weights=_tvm.ResNet18_Weights.IMAGENET1K_V1)
            self.out_channels = (64, 64, 128, 256, 512)
        elif name == 'resnet34':
            m = _tvm.resnet34(weights=_tvm.ResNet34_Weights.IMAGENET1K_V1)
            self.out_channels = (64, 64, 128, 256, 512)
        else:
            m = _tvm.resnet50(weights=_tvm.ResNet50_Weights.IMAGENET1K_V2)
            self.out_channels = (64, 256, 512, 1024, 2048)
        self.stem = nn.Sequential(m.conv1, m.bn1, m.relu); self.maxpool = m.maxpool
        self.layer1 = m.layer1; self.layer2 = m.layer2
        self.layer3 = m.layer3; self.layer4 = m.layer4
    def forward(self, x):
        x0 = self.stem(x); x1 = self.layer1(self.maxpool(x0))
        x2 = self.layer2(x1); x3 = self.layer3(x2)
        return x0, x1, x2, x3, self.layer4(x3)

class _UNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.encoder = _Enc(cfg.encoder)
        e0,e1,e2,e3,e4 = self.encoder.out_channels
        d = cfg.base_channels; cd = cfg.view_embed_dim; ua = cfg.use_attention_gates
        self.view_embed = nn.Embedding(cfg.view_count, cd)
        self.cond_mlp   = nn.Sequential(nn.Linear(cd,cd), nn.SiLU(inplace=True), nn.Linear(cd,cd))
        self.bottleneck = nn.Sequential(_CBA(e4,d), _CBA(d,d))
        self.up3 = _Up(d,e3,d//2,cd,ua); self.up2 = _Up(d//2,e2,d//4,cd,ua)
        self.up1 = _Up(d//4,e1,d//8,cd,ua); self.up0 = _Up(d//8,e0,d//8,cd,ua)
        self.head = nn.Sequential(_CBA(d//8,d//16), nn.Conv2d(d//16,1,1))
    def forward(self, x, vid):
        c = self.cond_mlp(self.view_embed(vid.long()))
        x0,x1,x2,x3,x4 = self.encoder(x)
        b = self.bottleneck(x4)
        d3=self.up3(b,x3,c); d2=self.up2(d3,x2,c)
        d1=self.up1(d2,x1,c); d0=self.up0(d1,x0,c)
        return self.head(F.interpolate(d0, scale_factor=2.0, mode='bilinear', align_corners=False))

def _load_unet(ckpt_path):
    ck       = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    cfg_raw  = ck.get('cfg', {})
    cfg_dict = _dc.asdict(cfg_raw) if _dc.is_dataclass(cfg_raw) else (dict(cfg_raw) if cfg_raw else {})
    m        = _UNet(_MCfg(**cfg_dict))
    m.load_state_dict(ck['model'], strict=True)
    return m.to(DEVICE).eval()

_VRE = _re.compile(r'^.+_view([1-5])\.png$')
def _vid(fn): m = _VRE.match(Path(fn).name); return int(m.group(1)) - 1 if m else 0
def _pil2t(img):
    return torch.from_numpy(np.asarray(img, np.float32).transpose(2,0,1) / 255.)
def _fg(img, thr=15):
    return (np.asarray(img.convert('L'), np.uint8) > thr).astype(np.float32)

@torch.no_grad()
def unet_hm(model, img_path):
    img   = Image.open(img_path).convert('RGB').resize((UNET_IMG_SIZE, UNET_IMG_SIZE), Image.BILINEAR)
    fg    = _fg(img)
    vid   = torch.tensor([_vid(Path(img_path).name)], dtype=torch.long, device=DEVICE)
    preds = []
    for fl in (['', 'h', 'v'] if ENS_USE_TTA else ['']):
        im = (img.transpose(Image.FLIP_LEFT_RIGHT) if fl == 'h' else
              img.transpose(Image.FLIP_TOP_BOTTOM) if fl == 'v' else img)
        p  = torch.sigmoid(model(_pil2t(im).unsqueeze(0).to(DEVICE), vid))[0,0].cpu().numpy()
        if fl == 'h': p = p[:, ::-1].copy()
        if fl == 'v': p = p[::-1, :].copy()
        preds.append(p)
    prob = np.mean(preds, 0) * fg
    return np.array(Image.fromarray((prob * 255).astype(np.uint8))
                    .resize((224, 224), Image.BILINEAR)) / 255.

print('Helpers ready.')

Helpers ready.


In [ ]:
# ── Build or load score cache ─────────────────────────────────────────────────
# Cache format: {CACHE_DIR}/{cls}.npz
#   pc     (N, 224, 224) float32 — calibrated PatchCore heatmaps
#   un     (N, 224, 224) float32 — UNet heatmaps (zeros if no checkpoint)
#   lb     (N, 224, 224) float32 — ground-truth masks
#   atypes (N,)          object  — 'good' or 'anomaly_xxx'
#   has_un (1,)          bool
#
# To invalidate the cache for a class, delete its .npz file in CACHE_DIR.

cache   = {}
need_pc = []

for cls in classes:
    fp = CACHE_DIR / f'{cls}.npz'
    if fp.exists():
        d          = np.load(fp, allow_pickle=True)
        cache[cls] = {k: d[k] for k in d.files}
        n          = len(cache[cls]['lb'])
        has_un     = bool(cache[cls].get('has_un', [True])[0])
        print(f'  {cls}: loaded ({n} imgs, UNet={has_un})')
    else:
        need_pc.append(cls)
        print(f'  {cls}: cache missing — will compute')

if need_pc:
    print(f'\nLoading DINOv2 ...')
    dino = build_dino()

    print(f'\nBuilding feature banks for {need_pc} ...')
    pca, banks = build_banks_for(need_pc, dino)

    for cls in need_pc:
        print(f'\n── {cls} ──────────────────────────────────────')
        items      = val_items_for(cls)
        img_paths  = [str(item[0]) for item in items]
        mask_paths = [item[1]      for item in items]
        atypes     = [item[2]      for item in items]

        train_paths = train_normal_paths_for(cls)
        print(f'  Calibrating on {len(train_paths)} training normals ...')
        hi = calib_hi_for(train_paths, dino, pca, banks[cls])
        print(f'  calib_hi={hi:.4f}')

        print(f'  Scoring {len(img_paths)} val images with PatchCore ...')
        pc_hms = score_paths_pc(img_paths, dino, pca, banks[cls], hi)

        ckpt = Path(UNET_DIR) / cls / 'best.pt'
        if ckpt.exists():
            print('  Running UNet inference ...')
            unet   = _load_unet(str(ckpt))
            un_hms = [unet_hm(unet, p) for p in tqdm(img_paths, desc='UNet', leave=False)]
            del unet;  torch.cuda.empty_cache()
            has_un = True
        else:
            print('  No UNet checkpoint — storing zeros')
            un_hms = [np.zeros((224, 224), np.float32) for _ in img_paths]
            has_un = False

        lb_hms = [load_mask(str(mp)) if mp is not None else np.zeros((224, 224), np.float32)
                  for mp in mask_paths]

        data = dict(
            pc     = np.stack(pc_hms).astype(np.float32),
            un     = np.stack(un_hms).astype(np.float32),
            lb     = np.stack(lb_hms).astype(np.float32),
            atypes = np.array(atypes, dtype=object),
            has_un = np.array([has_un]),
        )
        np.savez_compressed(CACHE_DIR / f'{cls}.npz', **data)
        cache[cls] = data
        print(f'  Saved: {len(img_paths)} images')

    del dino, pca, banks;  torch.cuda.empty_cache()

print('\nAll classes ready.')

  class_03: cache missing — will compute

Loading DINOv2 ...

Building feature banks for ['class_03'] ...


class_03 feats:   0%|          | 0/32 [00:00<?, ?it/s]

  class_03: 2016 imgs -> 49,984 coreset patches
Fitting PCA 3072->384 on 49,984 patches ... done
  class_03: bank torch.Size([49984, 384])

── class_03 ──────────────────────────────────────
  Calibrating on 2016 training normals ...
  calib_hi=7348.3823
  Scoring 524 val images with PatchCore ...


PC score:   0%|          | 0/9 [00:00<?, ?it/s]

  Running UNet inference ...


UNet:   0%|          | 0/524 [00:00<?, ?it/s]

  Saved: 524 images

All classes ready.


In [ ]:
# ── Strategy benchmark ────────────────────────────────────────────────────────
# Flatten cached heatmaps into global + per-class 1-D pixel arrays.
# Alpha sweep on subsample; all strategies evaluated on the full array.

cls_flat = {}   # cls -> (pc_1d, un_1d, lb_1d)
all_pc, all_un, all_lb = [], [], []

for cls in classes:
    d  = cache[cls]
    pc = d['pc'].reshape(-1).astype(np.float32)
    un = d['un'].reshape(-1).astype(np.float32)
    lb = d['lb'].reshape(-1).astype(np.float32)
    cls_flat[cls] = (pc, un, lb)
    all_pc.append(pc);  all_un.append(un);  all_lb.append(lb)

pc_g = np.concatenate(all_pc)
un_g = np.concatenate(all_un)
lb_g = np.concatenate(all_lb)
print(f'Total pixels: {len(lb_g):,}  |  anomalous: {lb_g.sum():,.0f}  ({100*lb_g.mean():.3f}%)')

# Per-class AP baseline
print('\n── Per-class AP (PC-only / UNet-only) ───────────────────────')
for cls, (pc_c, un_c, lb_c) in cls_flat.items():
    if lb_c.sum() == 0:
        print(f'  {cls}: no anomaly pixels'); continue
    ap_pc = average_precision_score(lb_c, pc_c)
    ap_un = average_precision_score(lb_c, un_c)
    print(f'  {cls:<20}  PC={ap_pc:.4f}  UNet={ap_un:.4f}')
ap_pc_g = average_precision_score(lb_g, pc_g)
ap_un_g = average_precision_score(lb_g, un_g)
print(f'  {"GLOBAL":<20}  PC={ap_pc_g:.4f}  UNet={ap_un_g:.4f}')

# Alpha sweep on pixel subsample
rng_s = np.random.default_rng(0)
idx_s = rng_s.choice(len(lb_g), max(1, int(len(lb_g) * SWEEP_FRAC)), replace=False)
pc_s, un_s, lb_s = pc_g[idx_s], un_g[idx_s], lb_g[idx_s]

print('\n── Alpha sweep (global, subsampled) ─────────────────────────')
best_alpha, best_wa_ap_s = 0.5, 0.0
for alpha in np.linspace(0, 1, ALPHA_STEPS):
    ap = average_precision_score(lb_s, alpha * pc_s + (1 - alpha) * un_s)
    if ap > best_wa_ap_s:
        best_wa_ap_s, best_alpha = ap, float(alpha)
        print(f'  α={alpha:.2f}  AP≈{ap:.4f}  <--')

# Evaluate all strategies on full array
_a = best_alpha
strategies = {
    f'weighted_avg (α={best_alpha:.2f})': (
        average_precision_score(lb_g, best_alpha * pc_g + (1 - best_alpha) * un_g),
        lambda p, u, a=_a: a * p + (1 - a) * u),
    'max': (
        average_precision_score(lb_g, np.maximum(pc_g, un_g)),
        np.maximum),
    'geometric_mean': (
        average_precision_score(lb_g, np.sqrt(pc_g * un_g)),
        lambda p, u: np.sqrt(p * u)),
    'cascaded (un + pc*(1-un))': (
        average_precision_score(lb_g, un_g + pc_g * (1 - un_g)),
        lambda p, u: u + p * (1 - u)),
}

print('\n── Strategy comparison (full array) ─────────────────────────')
print(f'  {"PC-only":<36}  AP={ap_pc_g:.4f}')
print(f'  {"UNet-only":<36}  AP={ap_un_g:.4f}')
best_strategy = max(strategies, key=lambda k: strategies[k][0])
best_fuse     = strategies[best_strategy][1]
best_ap       = strategies[best_strategy][0]
for name, (ap, _) in sorted(strategies.items(), key=lambda x: -x[1][0]):
    tag = '  <-- best' if name == best_strategy else ''
    print(f'  {name:<36}  AP={ap:.4f}{tag}')
print(f'\nBest: {best_strategy}  (AP={best_ap:.4f})')

Total pixels: 26,292,224  |  anomalous: 8,365  (0.032%)

── Per-class AP (PC-only / UNet-only) ───────────────────────
  class_03              PC=0.1666  UNet=0.0038
  GLOBAL                PC=0.1666  UNet=0.0038

── Alpha sweep (global, subsampled) ─────────────────────────
  α=0.00  AP≈0.0032  <--
  α=0.03  AP≈0.0066  <--
  α=0.05  AP≈0.0095  <--
  α=0.08  AP≈0.0120  <--
  α=0.10  AP≈0.0141  <--
  α=0.12  AP≈0.0158  <--
  α=0.15  AP≈0.0175  <--
  α=0.18  AP≈0.0198  <--
  α=0.20  AP≈0.0225  <--
  α=0.23  AP≈0.0255  <--
  α=0.25  AP≈0.0286  <--
  α=0.28  AP≈0.0320  <--
  α=0.30  AP≈0.0353  <--
  α=0.33  AP≈0.0388  <--
  α=0.35  AP≈0.0426  <--
  α=0.38  AP≈0.0469  <--
  α=0.40  AP≈0.0517  <--
  α=0.43  AP≈0.0573  <--
  α=0.45  AP≈0.0639  <--
  α=0.48  AP≈0.0721  <--
  α=0.50  AP≈0.0820  <--
  α=0.53  AP≈0.0943  <--
  α=0.55  AP≈0.1100  <--
  α=0.58  AP≈0.1281  <--
  α=0.60  AP≈0.1459  <--
  α=0.62  AP≈0.1619  <--
  α=0.65  AP≈0.1742  <--
  α=0.68  AP≈0.1829  <--
  α=0.70  AP≈0.1893  <--

In [ ]:
# ── Per-class alpha sweep ─────────────────────────────────────────────────────
print(f'Baseline: {best_strategy}  AP={best_ap:.4f}\n')
print(f'  {"class":<20}  {"α*":<6}  AP (full)')
print('  ' + '─' * 38)

rng_s      = np.random.default_rng(0)
cls_alphas = {}
fuse_all, lb_all = [], []

for cls, (pc_c, un_c, lb_c) in cls_flat.items():
    n   = len(lb_c)
    idx = rng_s.choice(n, max(1, int(n * SWEEP_FRAC)), replace=False)
    pcs, uns, lbs = pc_c[idx], un_c[idx], lb_c[idx]

    best_ap_c, best_a_c = 0.0, 0.5
    if lbs.sum() > 0:
        for alpha in np.linspace(0, 1, ALPHA_STEPS):
            ap = average_precision_score(lbs, alpha * pcs + (1 - alpha) * uns)
            if ap > best_ap_c:
                best_ap_c, best_a_c = ap, float(alpha)

    full_ap = (average_precision_score(lb_c, best_a_c * pc_c + (1 - best_a_c) * un_c)
               if lb_c.sum() > 0 else 0.0)
    cls_alphas[cls] = best_a_c
    print(f'  {cls:<20}  α*={best_a_c:.2f}  AP={full_ap:.4f}')
    fuse_all.append(best_a_c * pc_c + (1 - best_a_c) * un_c)
    lb_all.append(lb_c)

pca_ap = average_precision_score(np.concatenate(lb_all), np.concatenate(fuse_all))
print(f'\n  Global pixel-AP (per-class α): {pca_ap:.4f}  ({pca_ap - best_ap:+.4f} vs best)')
if pca_ap > best_ap:
    best_ap       = pca_ap
    best_strategy = 'per_class_alpha'
    print('  -> New best!')
print(f'\n  cls_alphas: {cls_alphas}')

Baseline: weighted_avg (α=0.78)  AP=0.1965

  class                 α*      AP (full)
  ──────────────────────────────────────
  class_03              α*=0.78  AP=0.1965

  Global pixel-AP (per-class α): 0.1965  (+0.0000 vs best)

  cls_alphas: {'class_03': 0.775}


In [ ]:
# ── Gamma sweep — pc^γ × all strategies ──────────────────────────────────────
# Inner sweeps on pixel subsample; only the best (γ, strategy) re-evaluated on full array.

print(f'Baseline: {best_strategy}  AP={best_ap:.4f}\n')
print(f'  {"γ":<6}  {"best strategy":<36}  AP≈ (subsampled)')
print('  ' + '─' * 62)

rng_s = np.random.default_rng(0)
idx_s = rng_s.choice(len(lb_g), max(1, int(len(lb_g) * SWEEP_FRAC)), replace=False)
pc_s, un_s, lb_s = pc_g[idx_s], un_g[idx_s], lb_g[idx_s]

best_cand_ap_s, best_cand = -1.0, None

for g in GAMMA_VALUES:
    pc_gs = np.power(pc_s, g)

    wa_ap_g, wa_alpha_g = 0.0, 0.5
    for alpha in np.linspace(0, 1, ALPHA_STEPS):
        ap = average_precision_score(lb_s, alpha * pc_gs + (1 - alpha) * un_s)
        if ap > wa_ap_g:
            wa_ap_g, wa_alpha_g = ap, float(alpha)

    _ag = wa_alpha_g
    strats_g = {
        f'weighted_avg (α={wa_alpha_g:.2f})': (wa_ap_g,
            lambda p, u, a=_ag, gg=g: a * np.power(p, gg) + (1 - a) * u),
        'max': (average_precision_score(lb_s, np.maximum(pc_gs, un_s)),
            lambda p, u, gg=g: np.maximum(np.power(p, gg), u)),
        'geometric_mean': (average_precision_score(lb_s, np.sqrt(pc_gs * un_s)),
            lambda p, u, gg=g: np.sqrt(np.power(p, gg) * u)),
        'cascaded': (average_precision_score(lb_s, un_s + pc_gs * (1 - un_s)),
            lambda p, u, gg=g: u + np.power(p, gg) * (1 - u)),
    }

    best_g_name = max(strats_g, key=lambda k: strats_g[k][0])
    ap_g_s      = strats_g[best_g_name][0]
    tag         = '  *' if ap_g_s > best_cand_ap_s else ''
    print(f'  γ={g:<5.2f}  {best_g_name:<36}  AP≈{ap_g_s:.4f}{tag}')
    if ap_g_s > best_cand_ap_s:
        best_cand_ap_s = ap_g_s
        best_cand      = (g, f'γ={g:.2f} / {best_g_name}', strats_g[best_g_name][1])

if best_cand is not None:
    g_best, name_best, fn_best = best_cand
    ap_full = average_precision_score(lb_g, fn_best(pc_g, un_g))
    print(f'\nBest candidate "{name_best}": full AP={ap_full:.4f}  ({ap_full-best_ap:+.4f})')
    if ap_full > best_ap:
        best_ap       = ap_full
        best_strategy = name_best
        best_fuse     = fn_best
        print('  -> NEW BEST')
    else:
        print(f'  -> No improvement over {best_ap:.4f}')

Baseline: weighted_avg (α=0.78)  AP=0.1965

  γ       best strategy                         AP≈ (subsampled)
  ──────────────────────────────────────────────────────────────
  γ=0.50   weighted_avg (α=0.88)                 AP≈0.1985  *
  γ=0.70   weighted_avg (α=0.83)                 AP≈0.1984
  γ=0.80   weighted_avg (α=0.80)                 AP≈0.1979
  γ=1.00   weighted_avg (α=0.78)                 AP≈0.1975
  γ=1.20   weighted_avg (α=0.75)                 AP≈0.1966
  γ=1.50   weighted_avg (α=0.75)                 AP≈0.1947
  γ=2.00   weighted_avg (α=0.80)                 AP≈0.1929
  γ=2.50   weighted_avg (α=0.80)                 AP≈0.1895
  γ=3.00   weighted_avg (α=0.85)                 AP≈0.1862

Best candidate "γ=0.50 / weighted_avg (α=0.88)": full AP=0.1974  (+0.0010)
  -> NEW BEST


In [ ]:
# ── Morphological post-processing ─────────────────────────────────────────────
# Operates on 2-D heatmaps (before flattening) so spatial structure is preserved.
# Tests median filter and min-area connected-component suppression.

def apply_cc_filter(hm, min_area, threshold=0.3):
    """Zero out connected components smaller than min_area pixels."""
    labeled, n = cc_label(hm >= threshold)
    out = hm.copy()
    for i in range(1, n + 1):
        if (labeled == i).sum() < min_area:
            out[labeled == i] = 0.0
    return out

def eval_postproc(fused_stack, lb_stack, morph_size=None, min_area=0):
    """fused_stack: (N,224,224), lb_stack: (N,224,224). Returns global AP."""
    if morph_size is None and min_area == 0:
        return average_precision_score(lb_stack.reshape(-1), fused_stack.reshape(-1))
    processed = np.empty_like(fused_stack)
    for i in range(len(fused_stack)):
        hm = fused_stack[i]
        if morph_size:
            hm = median_filter(hm, size=morph_size)
        if min_area > 0:
            hm = apply_cc_filter(hm, min_area)
        processed[i] = hm
    return average_precision_score(lb_stack.reshape(-1), processed.reshape(-1))

# Build fused 2-D stacks using best_fuse (works on any numpy shape)
all_fused_2d, all_lb_2d = [], []
for cls in classes:
    d  = cache[cls]
    pc = d['pc'];  un = d['un'];  lb = d['lb']
    fused_cls = np.stack([best_fuse(pc[i], un[i]) for i in range(len(pc))])
    all_fused_2d.append(fused_cls)
    all_lb_2d.append(lb)
all_fused_2d = np.concatenate(all_fused_2d)
all_lb_2d    = np.concatenate(all_lb_2d)

ap_base = eval_postproc(all_fused_2d, all_lb_2d)
print(f'Fused baseline (no post-proc): AP={ap_base:.4f}  (strategy: {best_strategy})')

print('\n── Median filter ────────────────────────────────────────────')
print(f'  {"size":<8}  AP       delta')
for sz in MORPH_SIZES:
    ap = eval_postproc(all_fused_2d, all_lb_2d, morph_size=sz)
    print(f'  {sz:<8}  {ap:.4f}  {ap-ap_base:+.4f}')

print('\n── Min-area CC filter (threshold=0.3) ───────────────────────')
print(f'  {"min_area":<10}  AP       delta')
for ma in MIN_AREAS:
    ap = eval_postproc(all_fused_2d, all_lb_2d, min_area=ma)
    print(f'  {ma:<10}  {ap:.4f}  {ap-ap_base:+.4f}')

print('\n── Combined grid search ─────────────────────────────────────')
best_combo_ap, best_combo = ap_base, (None, 0)
rows_combo = []
for sz in [None] + MORPH_SIZES:
    for ma in MIN_AREAS:
        if sz is None and ma == 0:
            continue
        ap = eval_postproc(all_fused_2d, all_lb_2d, morph_size=sz, min_area=ma)
        rows_combo.append({'morph_size': sz, 'min_area': ma, 'AP': ap, 'delta': ap - ap_base})
        if ap > best_combo_ap:
            best_combo_ap, best_combo = ap, (sz, ma)

combo_df = pd.DataFrame(rows_combo).sort_values('AP', ascending=False)
print(combo_df.head(10).to_string(index=False))
print(f'\nBest: morph_size={best_combo[0]}  min_area={best_combo[1]}  AP={best_combo_ap:.4f}  ({best_combo_ap-ap_base:+.4f})')

Fused baseline (no post-proc): AP=0.1974  (strategy: γ=0.50 / weighted_avg (α=0.88))

── Median filter ────────────────────────────────────────────
  size      AP       delta
  3         0.2001  +0.0027
  5         0.2047  +0.0072
  7         0.2071  +0.0096

── Min-area CC filter (threshold=0.3) ───────────────────────
  min_area    AP       delta
  0           0.1974  +0.0000
  25          0.1974  -0.0000
  50          0.1974  +0.0000
  100         0.1974  +0.0000
  200         0.1974  +0.0000

── Combined grid search ─────────────────────────────────────
 morph_size  min_area       AP    delta
        7.0        25 0.207080 0.009645
        7.0        50 0.207080 0.009645
        7.0       100 0.207080 0.009645
        7.0       200 0.207080 0.009645
        7.0         0 0.207080 0.009645
        5.0        25 0.204679 0.007244
        5.0        50 0.204679 0.007244
        5.0       100 0.204679 0.007244
        5.0         0 0.204679 0.007244
        5.0       200 0.204679 0.007

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
rows = [
    {'strategy': 'PC-only',    'AP': ap_pc_g},
    {'strategy': 'UNet-only',  'AP': ap_un_g},
]
for name, (ap, _) in strategies.items():
    rows.append({'strategy': name, 'AP': ap})
rows.append({'strategy': 'per_class_alpha', 'AP': pca_ap})
if 'name_best' in dir() and 'ap_full' in dir():
    rows.append({'strategy': f'gamma ({name_best})', 'AP': ap_full})
if best_combo[0] is not None or best_combo[1] > 0:
    rows.append({'strategy': f'postproc (morph={best_combo[0]}, min_area={best_combo[1]})',
                 'AP': best_combo_ap})

df = pd.DataFrame(rows).sort_values('AP', ascending=False).reset_index(drop=True)
df['delta_vs_PC'] = df['AP'] - ap_pc_g
df['delta_vs_PC'] = df['delta_vs_PC'].map('{:+.4f}'.format)
df['AP']          = df['AP'].map('{:.4f}'.format)

print('══ Benchmark Results ════════════════════════════════════════')
print(f'  Classes: {classes}')
print()
print(df.to_string(index=False))
print(f'\nBest overall: {best_strategy}  AP={best_ap:.4f}')

══ Benchmark Results ════════════════════════════════════════
  Classes: ['class_03']

                              strategy     AP delta_vs_PC
       postproc (morph=7, min_area=25) 0.2071     +0.0405
gamma (γ=0.50 / weighted_avg (α=0.88)) 0.1974     +0.0309
                 weighted_avg (α=0.78) 0.1965     +0.0299
                       per_class_alpha 0.1965     +0.0299
                               PC-only 0.1666     +0.0000
                                   max 0.1134     -0.0531
             cascaded (un + pc*(1-un)) 0.1002     -0.0663
                        geometric_mean 0.0440     -0.1226
                             UNet-only 0.0038     -0.1628

Best overall: γ=0.50 / weighted_avg (α=0.88)  AP=0.1974
